# NB4 · Açıklanabilirlik

**Üretken Yapay Zekâ Araçları ile Klinik Karar Destek Sistemleri Geliştirilmesi**  
Sağlık Bilimlerinde Teknoloji ve Yapay Zekâ Okuryazarlığı Eğitimi · Akdeniz Üniversitesi · 18 Eylül 2026

Prof. Dr. Utku Köse · Süleyman Demirel Üniversitesi, Bilgisayar Mühendisliği Bölümü  
Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM) Müdürü · utkukose@sdu.edu.tr

---

Buradaki asıl iş kodun doğruluğunu değil, açıklamanın sınırlarını değerlendirmektir.


## Hazırlık


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
for modul in ['checks.py', 'evaluate.py', 'explain.py', 'safety.py',
              'mimic_web.py', 'pipeline.py']:
    urllib.request.urlretrieve(f'{REPO}/workshop/{modul}', modul)

import numpy as np
import pandas as pd
import checks, evaluate as ev, explain as ex, safety as sf

checks.LANG = ev.LANG = sf.LANG = 'tr'


In [ ]:
# Sabit hücre. Önceki defterlerin çıktısını yeniden kurar.
import pipeline as pl

durum = pl.prepare(verbose=False)
model = durum['model']
test = durum['test']
ozellikler = durum['features']
olasilik = durum['probabilities']
y_test = durum['y_test']
esik = ev.threshold_for_sensitivity(y_test, olasilik, target=0.80)
print(f'Model ve tahminler hazır. Çalışma eşiği: {esik:.3f}')


---

## Adım 1 · Küresel açıklama

Modeli genel olarak hangi özniteliklerin sürüklediği sorulacaktır. Permütasyon önemi
bunu model bağımsız biçimde ölçer: Bir öznitelik karıştırıldığında başarımın ne kadar
düştüğüne bakar.

Sonuçta yayılım değeri, ortalama değer kadar önemlidir. Küçük bir kohortta tekrarlar
arasındaki yayılım, komşu öznitelikler arasındaki farktan büyük olur. Böyle bir durumda
sıralamanın kendisi kararsızdır ve bir sıralama olarak sunulamaz.


### İstem 1

```
model adında eğitilmiş bir scikit-learn Pipeline ve test adında bir DataFrame var.
ozellikler listesi modelin kullandığı sütun adlarını içeriyor. Hedef sütunu hedef.

Permütasyon önemi hesaplayan tek bir Python hücresi yaz. Puanlama ölçütü roc_auc,
tekrar sayısı 20 olsun.

KABUL ÖLÇÜTLERİ
onem adında bir DataFrame üret. Sütunları: oznitelik, onem, yayilim
onem sütununa göre azalan sırada sırala ve ilk on beş satırı ekrana yaz.
```


In [ ]:
# Ürettiğiniz kodu bu hücreye yapıştırınız ve çalıştırınız.


### Kontrol 1


In [ ]:
checks.check_columns(onem, required=['oznitelik', 'onem', 'yayilim'], name='onem')

kararsiz = (onem['onem'] <= 2 * onem['yayilim']).sum()
print(f'\nSıralaması kararsız öznitelik sayısı: {kararsiz} / {len(onem)}')
print('Bu sayı yüksekse öznitelik sıralamasını bir bulgu olarak sunmayınız.')


---

## Adım 2 · Tek vaka açıklaması

Üç vaka seçilecektir: Bir doğru pozitif, bir yanlış pozitif ve bir yanlış negatif.

Önemli olan yanlış pozitiftir. Yanlış bir tahmini makul gösteren bir açıklama,
açıklanabilirliğin hatayı aklama mekanizmasıdır. Bunu bir kez görmek, kavramın tanımını
öğrenmekten daha öğreticidir.

Lojistik regresyonda katkılar yaklaşık değil kesindir. Log odds değeri, katsayı çarpı
öznitelik değerlerinin toplamıdır; dolayısıyla tek bir tahmindeki her özniteliğin katkısı
doğrudan okunabilir. Doğrusal olmayan bir modele geçildiğinde bu ayrıştırma kaybolur ve
SHAP gibi sonradan yapılan bir yaklaşıklama gerekir.


### İstem 2

```
Aynı model ve test kümesiyle çalışıyorum. olasilik dizisi test kümesindeki her satırın
pozitif sınıf olasılığını içeriyor. esik değişkeni karar eşiğini veriyor.

Tek bir Python hücresi yaz:
1. Bir doğru pozitif, bir yanlış pozitif ve bir yanlış negatif vakanın satır
   numarasını bul.
2. Her vaka için modelin o tahmine katkı veren özniteliklerini hesapla. Model doğrusal
   olduğu için katsayı çarpı dönüştürülmüş öznitelik değeri kullan.
3. Her vaka için ilk sekiz katkıyı, gerçek öznitelik değerleriyle birlikte yazdır.

KABUL ÖLÇÜTLERİ
vakalar adında bir sözlük üret. Anahtarları dogru_pozitif, yanlis_pozitif ve
yanlis_negatif olsun; değerleri satır numarası olsun. Vaka bulunamazsa değer None
olsun.
```


In [ ]:
# Ürettiğiniz kodu bu hücreye yapıştırınız ve çalıştırınız.


### Kontrol 2


In [ ]:
for ad in ['dogru_pozitif', 'yanlis_pozitif', 'yanlis_negatif']:
    durum_vaka = vakalar.get(ad)
    print(f'{ad:<16} {"bulunamadı" if durum_vaka is None else f"satır {durum_vaka}"}')


---

## Sabit çözümleme

Aşağıdaki hücreler defterin sabit bölümüdür. Katkıları kesin biçimde hesaplar ve
sonucun bir özdeşlik olduğunu gösterir: Katkıların toplamı artı kesişim log odds
değerini verir, onun sigmoidi ise modelin ürettiği olasılığa eşittir. Bir yaklaşıklamada
bu eşitlik tutmaz.


In [ ]:
vaka = vakalar.get('yanlis_pozitif') or vakalar.get('dogru_pozitif')
aciklama = ex.explain_case(model, test[ozellikler], vaka, top=10**6)

katkilardan = 1 / (1 + np.exp(-aciklama['log_odds']))
print(f"Katkılardan hesaplanan olasılık : {katkilardan:.10f}")
print(f"Modelin ürettiği olasılık       : {aciklama['probability']:.10f}")
print(f"Fark                            : {abs(katkilardan - aciklama['probability']):.2e}")


In [ ]:
for ad in ['dogru_pozitif', 'yanlis_pozitif', 'yanlis_negatif']:
    vaka = vakalar.get(ad)
    if vaka is None:
        continue
    aciklama = ex.explain_case(model, test[ozellikler], vaka, top=8)
    ex.plot_case(aciklama, title=f"{ad} · olasılık {aciklama['probability']:.3f}")


## Açıklamanın eleştirisi

Üç grafiğe bakarak aşağıdaki soruları kendiniz cevaplayınız.

Yüksek katkılı özniteliklerden biri klinik bir sinyal yerine kaydın yapılış biçiminden
kaynaklanan bir artefakt olabilir mi? Bu kohortta ölçüm sayısı sütunları buna adaydır.
Bir hastadan ilk altı saatte çok sayıda ölçüm alınmış olması, o hastanın zaten ağır
kabul edildiğinin göstergesi olabilir. Bu klinik bir bulgu değil, bakım yoğunluğu
vekilidir.

Yanlış pozitif açıklamasını okuyan bir klinisyen tahminin makul olduğuna ikna olur
muydu? Olursa bu bir başarı değil sorundur.

Katkılardan hangileri nedensel bir iddia gibi okunabilir? Düşük sistolik basıncın yüksek
katkı vermesi, basıncı yükseltmenin yatış süresini kısaltacağı anlamına gelmez.

Aynı soruları yapay zekâ aracına da yöneltiniz ve cevaplarını kendi cevaplarınızla
karşılaştırınız. Aracın kendi ürettiği açıklamayı eleştirmesi istendiğinde, özet
istenmesine kıyasla belirgin biçimde daha eleştirel davrandığı görülecektir.
---

**Uyarı.** Bu defterde üretilen hiçbir çıktı doğrulanmış bir klinik araç değildir.
MIMIC-IV demo verisi tek bir Amerikan hastanesinden gelmektedir ve Türkiye'deki bir
yoğun bakım popülasyonunu temsil etmez. Materyal öğretim amaçlıdır.
